# Big Test Cleaning and Visualization

Thise file in will load in the data from "THE BIG TEST" dataset whoch collects preference data on 24 beverages a session across several sessions over multiple years.

In [34]:
from dotenv import load_dotenv
import os
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

## STEP 1: Connecting to the SQL database


In [35]:
# Setting paths to SQL server based on your local storage location
load_dotenv(r"C:\Users\thaug\culinary_preferences\.env") # Load tour own personal .env where personal strings are stored

# Checking to make sure that the connection works - should yield culinary_preferences
print(os.getenv("MYSQL_DATABASE"))  

culinary_preferences


Format for example .env file:

MYSQL_HOST=localhost
MYSQL_USER=root
MYSQL_PASSWORD=YOUR_PASSWORD
MYSQL_DATABASE=culinary_preferences

RAW_DATA_DIR=C:\Users\YOUR_USER\culinary_preferences\data\raw
PROCESSED_DATA_DIR=C:\Users\YOUR_USER\culinary_preferences\data\processed

We then use the environment inputs to create an engine for the SQL database

In [36]:
engine = create_engine(
    f"mysql+mysqlconnector://"
    f"{os.getenv('MYSQL_USER')}:"
    f"{os.getenv('MYSQL_PASSWORD')}@"
    f"{os.getenv('MYSQL_HOST')}/"
    f"{os.getenv('MYSQL_DATABASE')}"
)  

# Testing that the enginve connection works - should yield (1,)
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print(result.fetchone())

(1,)


We then display the tables using pandas

In [37]:
tables = pd.read_sql(
    """
    SHOW TABLES;
    """,
    engine
)

tables

,Tables_in_culinary_preferences
0,beverage_rankings
1,beverage_type
2,beverages
3,country_of_origin
4,people
5,years


Showing the current schema of our database

In [38]:
schema = pd.read_sql(
    """
    SELECT
        table_name,
        column_name,
        data_type,
        is_nullable,
        column_key
    FROM information_schema.columns
    WHERE table_schema = 'culinary_preferences'
    ORDER BY table_name, ordinal_position;
    """,
    engine
)

schema

,TABLE_NAME,COLUMN_NAME,DATA_TYPE,IS_NULLABLE,COLUMN_KEY
0,beverage_rankings,id,int,NO,PRI
1,beverage_rankings,year_id,int,NO,MUL
2,beverage_rankings,person_id,int,NO,MUL
3,beverage_rankings,list_position,int,NO,
4,beverage_rankings,beverage_id,int,NO,MUL
5,beverage_rankings,score,decimal,YES,
6,beverage_type,id,int,NO,PRI
7,beverage_type,beverage_type_name,varchar,NO,UNI
8,beverages,id,int,NO,PRI
9,beverages,beverage_name,varchar,NO,UNI


Displaying the foreing keys connecting the tables

In [39]:
foreign_keys = pd.read_sql(
    """
    SELECT
        table_name,
        column_name,
        referenced_table_name,
        referenced_column_name
    FROM information_schema.key_column_usage
    WHERE table_schema = 'culinary_preferences'
      AND referenced_table_name IS NOT NULL;
    """,
    engine
)

foreign_keys

,TABLE_NAME,COLUMN_NAME,REFERENCED_TABLE_NAME,REFERENCED_COLUMN_NAME
0,beverage_rankings,year_id,years,id
1,beverage_rankings,person_id,people,id
2,beverage_rankings,beverage_id,beverages,id
3,beverages,beverage_type_id,beverage_type,id
4,beverages,country_of_origin_id,country_of_origin,id


## STEP 2: Load in the data from the Excell sheets

In [40]:
# Setting variables
raw_folder = Path(os.getenv("RAW_DATA_DIR")) # Setting the raw folder
excel_file = raw_folder / "DEN STORE TESTEN.xlsx" # Name of the raw data file

In [ ]:
# Loading in the data
data = pd.read_excel(excel_file)
data.tail()

,Nummer,ØL,Matsiboy,Torbz,Hermz,Nur,Vedo,Gjennomsnitt
0,Nr. 1:,Ægir Happy Red Ale,50,50,20,49,18,37.4
1,Nr. 2:,Lofoten Pale Ale,48,47,28,45,33,40.2
2,Nr. 3:,Sour Suzy Berliner Weis,45,30,37,39,49,40.0
3,Nr. 4:,Nøisom juleøl,20,24,20,37,35,27.2
4,Nr. 5:,Kirin Ichiban,55,60,55,50,59,55.8
